In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/

In [ ]:
import pandas as pd
import numpy as np
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip"
NUM_SYNTHETIC_SAMPLES = 5
EPOCHS = 200

print(f"Attempting to load file: {FILE_PATH}")
df = pd.read_csv(FILE_PATH, compression='zip', low_memory=False)

df.columns = df.columns.str.strip()

print(f"Original data shape: {df.shape}")

print("Using the full dataset for training.")

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.fillna(df.mean(numeric_only=True), inplace=True)

print(f"Cleaned data shape (after filling NaNs): {df.shape}")

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

if 'Label' in df.columns:
    metadata.update_column('Label', sdtype='categorical')

synthesizers = {
    "CTGAN": CTGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "CopulaGAN": CopulaGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "TVAE": TVAESynthesizer(metadata, epochs=EPOCHS)
}

synthetic_datasets = {}

for name, synthesizer in synthesizers.items():
    print(f"\n--- Training {name} model ---")
    try:
        synthesizer.fit(df)
        print(f"{name} training complete.")

        print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
        samples = synthesizer.sample(NUM_SYNTHETIC_SAMPLES)
        synthetic_datasets[name] = samples
        print(f"{name} Synthetic Samples:\n", samples)
    except Exception as e:
        print(f"Error during {name} training or sampling: {e}")

print("\n--- Summary of Generated Synthetic Data ---")
for name, data in synthetic_datasets.items():
    print(f"\n{name} Synthetic Data (first 5 rows):")
    print(data.head())
    print(f"{name} Data Shape: {data.shape}")

No GPUs detected. TensorFlow will run on CPU.
Attempting to load file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip
Original data shape: (225745, 79)
Using the full dataset for training.
Cleaned data shape (after filling NaNs): (225745, 79)


/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(



--- Training CTGAN model ---


KeyboardInterrupt: 

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

file_path = "/content/UNSW_NB15_training-set.parquet"
print(f"Attempting to load file: {file_path}")
df = pd.read_parquet(file_path)
print(f"Original data shape: {df.shape}")

print(f"Using the full dataset for training. Current shape: {df.shape}")

discrete_columns = ['proto', 'service', 'state', 'attack_cat', 'label']
continuous_columns = [col for col in df.columns if col not in discrete_columns]

df[continuous_columns] = df[continuous_columns].apply(pd.to_numeric, errors='coerce')
df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)
df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean())

for col in discrete_columns:
    df[col] = df[col].astype(object).fillna('unknown')

print(f"Cleaned data shape (after filling NaNs): {df.shape}")
print("\nFirst 5 rows of cleaned data:")
print(df.head())

metadata = Metadata.detect_from_dataframe(df)
for col in discrete_columns:
    metadata.update_column(col, sdtype='categorical')

EPOCHS = 200

synthesizers = {
    "CTGAN": CTGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "CopulaGAN": CopulaGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "TVAE": TVAESynthesizer(metadata, epochs=EPOCHS, verbose=True)
}

synthetic_data = {}

for name, synth in synthesizers.items():
    print(f"\n--- Training {name} ---")
    synth.fit(df)
    print(f"{name} training complete.")

    print(f"\nGenerating 10 synthetic samples using {name}...")
    samples = synth.sample(10)
    synthetic_data[name] = samples
    print(f"\n{name} Synthetic Samples:\n", samples)

print("\n--- Summary of Generated Synthetic Data ---")
for name, data in synthetic_data.items():
    print(f"\n{name} Data (first 5 rows):")
    print(data.head())
    print(f"{name} Shape: {data.shape}")

In [ ]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "/content/KDDTrain+.txt.zip"
NUM_SYNTHETIC_SAMPLES = 8
EPOCHS = 200

columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
    'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count',
    'serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
    'dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'outcome','level'
]

if not os.path.exists(FILE_PATH):
    print(f"Error: The file '{FILE_PATH}' was not found. Please upload it to Colab.")
else:
    print(f"File '{FILE_PATH}' found. Proceeding with data loading.")

    df = pd.read_csv(FILE_PATH, header=None, names=columns)
    print(f"Original data shape: {df.shape}")

    print(f"Using the full dataset for training. Current shape: {df.shape}")

    discrete_columns = ['protocol_type', 'service', 'flag', 'outcome', 'level']
    continuous_columns = [col for col in df.columns if col not in discrete_columns]

    for col in continuous_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)

    df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean(numeric_only=True))

    for col in discrete_columns:
        df[col] = df[col].astype(str).fillna('unknown')

    print(f"Cleaned data shape (after preprocessing): {df.shape}")
    print("\nFirst 5 rows of cleaned data:")
    print(df.head())

    metadata = Metadata.detect_from_dataframe(df)
    for col in discrete_columns:
        metadata.update_column(col, sdtype='categorical')

    synthesizers = {
        "CTGAN": CTGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
        "CopulaGAN": CopulaGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
        "TVAE": TVAESynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True)
    }

    synthetic_datasets = {}

    for name, synth in synthesizers.items():
        print(f"\n--- Training {name} model ---")
        try:
            synth.fit(df)
            print(f"{name} training complete.")

            print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
            samples = synth.sample(NUM_SYNTHETIC_SAMPLES)
            synthetic_datasets[name] = samples
            print(f"{name} Synthetic Samples:\n", samples)
        except Exception as e:
            print(f"Error during {name} training or sampling: {e}")

    print("\n--- Summary of Generated Synthetic Data ---")
    for name, data in synthetic_datasets.items():
        print(f"\n{name} Data (first 5 rows):")
        print(data.head())
        print(f"{name} Shape: {data.shape}")

In [ ]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "/content/reduced_data_1.csv.zip"
NUM_SYNTHETIC_SAMPLES = 5
EPOCHS = 200

if not os.path.exists(FILE_PATH):
    print(f"Error: The file '{FILE_PATH}' was not found. Please upload it to Colab.")
else:
    print(f"File '{FILE_PATH}' found. Proceeding with data loading.")

    df = pd.read_csv(FILE_PATH, low_memory=False)
    print(f"Original data shape: {df.shape}")

    print(f"Using the full dataset for training. Current shape: {df.shape}")

    discrete_columns = [
        'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
        'state', 'attack', 'category', 'subcategory'
    ]

    continuous_columns = [col for col in df.columns if col not in discrete_columns]

    for col in continuous_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)

    df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean())

    for col in discrete_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).fillna('unknown')
        else:
            print(f"Warning: Discrete column '{col}' not found in DataFrame.")

    initial_rows = df.shape[0]
    df_clean = df.dropna().reset_index(drop=True)
    if df_clean.shape[0] < initial_rows:
        print(f"Warning: {initial_rows - df_clean.shape[0]} rows dropped due to remaining NaNs.")

    print(f"Cleaned data shape (after preprocessing and final NaN check): {df_clean.shape}")
    print("\nFirst 5 rows of cleaned data:")
    print(df_clean.head())

    if df_clean.shape[0] == 0:
        print("Error: Cleaned dataframe is empty after processing. Cannot train any synthesizers.")
    else:
        metadata = Metadata.detect_from_dataframe(df_clean)
        for col in discrete_columns:
            if col in df_clean.columns:
                metadata.update_column(col, sdtype='categorical')

        synthesizers = {
            "CTGAN": CTGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
            "CopulaGAN": CopulaGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
            "TVAE": TVAESynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True)
        }

        synthetic_datasets = {}

        for name, synth in synthesizers.items():
            print(f"\n--- Training {name} model ---")
            try:
                synth.fit(df_clean)
                print(f"{name} training complete.")

                print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
                samples = synth.sample(NUM_SYNTHETIC_SAMPLES)
                synthetic_datasets[name] = samples
                print(f"{name} Synthetic Samples:\n", samples)
            except Exception as e:
                print(f"Error during {name} training or sampling: {e}")

        print("\n--- Summary of Generated Synthetic Data ---")
        for name, data in synthetic_datasets.items():
            print(f"\n{name} Data (first 5 rows):")
            print(data.head())
            print(f"{name} Shape: {data.shape}")